# 1 · Getting Started

Deploy the Retail Shopping Assistant on hosted NVIDIA endpoints, hold a first
conversation, then take a tour: one live call to each component, a look at the
agent inside, and the checks to reach for when something is off.

| | |
|---|---|
| **Time** | about 20 minutes, most of it the first image build |
| **You need** | Docker with Compose, an NVIDIA API key, Python 3.10+ |
| **GPU** | none. Every model runs on a hosted endpoint |
| **Next** | [2 · Observability](2_Observability.ipynb) |

**How to use this notebook.** Run the cells in order. Each step says what to
**Run** or **Do**, then shows what **You should see**. Model replies vary from
run to run; the shape should match. When it does not, the **If not** line says
where to look.

To run the models on your own GPUs, see [docs/DEPLOYMENT.md](../docs/DEPLOYMENT.md).
The rest of this notebook works the same either way.

## 1. What you are deploying

![architecture](shopping-assistant-diagram.jpg)

| Service | Port | Owns |
|---|---|---|
| **nginx + web UI** | 3000 | the shopper's page |
| **chain-server** | 8009 | the agent: one turn, from message to reply |
| **catalog-retriever** | 8010 | product search and product details |
| **memory-retriever** | 8011, this machine only | carts, and what each conversation has shown |
| **guardrails** | 8012 | content-safety checks (optional, not covered here) |
| **milvus** (+ etcd, minio) | 19530 | the vector index of the catalog |
| **phoenix** + otel-collector | 6006 | a trace of every turn |

**One turn, end to end:**

1. The UI posts the shopper's message to the chain-server.
2. The agent picks the **skills** that fit (styling, cart, store policy, ...).
3. Those skills decide which **tools** it may call: a catalog search, a cart write.
4. It writes the reply from what the tools returned, and nothing else.
5. The reply streams back. The turn is recorded as a trace in Phoenix.

## 2. Check your machine

**Do**, once, in a terminal:

```bash
git clone https://github.com/NVIDIA-AI-Blueprints/retail-shopping-assistant.git
cd retail-shopping-assistant
docker login nvcr.io          # username: $oauthtoken   password: your API key
export NVIDIA_API_KEY="nvapi-..."
export EXPOSE_AGENT_DIAGNOSTICS=true   # development only: each turn reports its skills and tools
jupyter lab notebook/                  # start Jupyter from this shell, so it has both
```

**Run** the next cell. It checks Docker, Compose, and the ports the stack uses.

In [ ]:
import json, shutil, socket, subprocess
from helpers import *

print("docker:", shutil.which("docker") or "MISSING")
print(subprocess.run(["docker", "compose", "version"], capture_output=True, text=True).stdout.strip())
for port in (3000, 8009, 8010, 8011, 6006):
    with socket.socket() as s:
        print(f"port {port}:", "in use" if s.connect_ex(("127.0.0.1", port)) == 0 else "free")

**You should see** a Docker path, a Compose version, and every port `free`:

```
docker: /usr/bin/docker
Docker Compose version v2.29.1
port 3000: free
...
```

**If not:** a port `in use` *before* you deploy belongs to another program. Stop
it, or change that port in `docker-compose.yaml`. After deploying, every port
shows `in use`, which is expected.

## 3. Configure

`.env.example` sends every model to a hosted endpoint and reads your key from
the environment. Your copy, `.env`, is ignored by git.

**Run** the next cell. It creates `.env` if needed, then loads it the way
deploy will and checks that a key came through. It never prints the key.

In [ ]:
env_file = REPO / ".env"
if not env_file.exists():
    shutil.copy(REPO / ".env.example", env_file)
    print("created", env_file)

def loaded(check):
    return subprocess.run(["bash", "-c", f'. "{env_file}" && {check}']).returncode == 0

print("NVIDIA_API_KEY:", "set" if loaded('[ -n "$NVIDIA_API_KEY" ]') else "NOT SET")
print("EXPOSE_AGENT_DIAGNOSTICS:", "true" if loaded('[ "$EXPOSE_AGENT_DIAGNOSTICS" = true ]') else "false")

**You should see:**

```
NVIDIA_API_KEY: set
EXPOSE_AGENT_DIAGNOSTICS: true
```

**If not:** export both in the shell that started Jupyter (step 2), and restart
Jupyter. Or set them straight in `.env`. Diagnostics put each turn's tool calls
in its response. The tour, notebook 3's checks, and debugging all read them.
Leave them off in production: they contain tool arguments and internal product
ids.

Model routing, meaning which model serves which role, lives in
`shared/configs/models.yaml`. Keep the defaults for this notebook.

## 4. Deploy

**Run** the next cell. It validates the model routing, then builds and starts
every service. The first build takes several minutes.

In [ ]:
%%bash
cd .. && set -a && . ./.env && set +a
python -m pip install --quiet --user -r requirements-deploy.txt
python scripts/model_config.py show --validate
python scripts/model_config.py deploy --build 2>&1 | tail -5

**You should see** the resolved endpoint for each model role (no keys), then
Compose reporting containers `Started` or `Healthy`.

A one-shot `catalog-indexer` embeds the catalog into Milvus and exits. Search
waits for it, so the stack is ready a little after the containers start.

**Run** the next cell. It waits until the services answer `/ready`.

In [ ]:
wait_until_ready()

**You should see** a few `waiting:` lines, then:

```
All services ready: chain-server, catalog-retriever, memory-retriever
```

**If not:** see [step 7](#7.-When-something-is-off).

## 5. Hold a first conversation

`say()` posts one message to `/query/stream`, the endpoint the UI uses, and
collects the reply, the products shown, and the turn's diagnostics. `show()`
prints them. Keep one conversation id for the whole conversation: the agent
remembers what it showed, so "the first one" means something.

**Run:**

In [ ]:
chat = new_conversation()
turn = say("I need a black dress for a summer wedding, under $150", chat)
show(turn)

**You should see** a reply, a few products, and then the skills and tools the
turn used:

```
Here are the black dresses under $150 that came up for a summer wedding: ...
  - Black Satin Lace-Up Dress  $69.99
  - Vivacious Velvet Dress  $129.99
  - Black Polka-Dotted Slip Dress  $59.9
[skills ['outfit-styling', 'budget-shopping'] | tools ['activate_shopper_skills_tool', 'search_catalog_tool'] | 8.2s]
```

**Run** a follow-up in the same conversation:

In [ ]:
turn = say("Add the first one in size 6", chat)
show(turn)

**You should see** a confirmation. The agent resolves "the first one" to a
product it showed, then adds it:

```
Added the Black Satin Lace-Up Dress in size 6 to your cart. ...
[skills ['outfit-styling', 'cart-management'] | tools ['activate_shopper_skills_tool', 'resolve_conversation_products_tool', 'add_cart_items_tool'] | 12.6s]
```

**Try** the same in the browser at [http://localhost:3000](http://localhost:3000).
The UI calls these same APIs.

## 6. Tour the components

Each step makes one live call to one service.

### 6.1 chain-server: the agent

| Endpoint | Use |
|---|---|
| `POST /query/stream` | one shopper turn, streamed. This is what `say()` calls |
| `POST /query/timing` | the same turn, returned whole with timings |
| `GET /cart?cart_id=` | the cart behind a browser's cart handle |
| `GET /capabilities` | which model serves each role, and what media is accepted |
| `GET /shopper-profiles` | the demo shoppers offered in the UI |
| `GET /health`, `GET /ready` | alive, and able to serve |

**Run:**

In [ ]:
caps = get(f"{CHAIN_SERVER}/capabilities")
for role, model in caps["models"].items():
    print(f"{role:16} {model['source']:9} {model['model']}")
print("media:", {k: caps["media_input"][k] for k in ("enabled", "vlm_enabled", "max_images_per_turn")})

**You should see** every role on a hosted `endpoint`:

```
app_llm          endpoint  nvidia/nvidia/nemotron-3.5-super-text-preview
vlm              endpoint  nvidia/nvidia/nemotron-3-nano-omni-30b-a3b-reasoning
text_embedding   endpoint  nvidia/nemotron-3-embed-1b
image_embedding  endpoint  nvidia/nvclip
content_safety   endpoint  nvidia/llama-3.1-nemoguard-8b-content-safety
topic_control    endpoint  nvidia/llama-3.1-nemoguard-8b-topic-control
media: {'enabled': True, 'vlm_enabled': True, 'max_images_per_turn': 1}
```

### 6.2 catalog-retriever: search, with no LLM

The catalog only retrieves: it embeds, searches vectors, applies hard filters,
and orders the results. Interpreting what the shopper meant is the agent's job.

`/capabilities` is the catalog describing itself. The agent's search tool is
generated from it, so a different catalog brings its own filters without a
code change.

**Run:**

In [ ]:
catalog = get(f"{CATALOG}/capabilities")
print(catalog["product_count"], "products; modes:", catalog["retrieval_modes"])
for category, spec in catalog["taxonomy"]["categories"].items():
    print(f"  {category:9} {', '.join(spec['subcategories'])}")
print("filters:", ", ".join(catalog["filters"]))

**You should see:**

```
215 products; modes: ['text']
  apparel   blouses, camisoles, dresses, jumpsuits, skirts, sweaters
  bags      clutches, crossbody_bags, satchels, shoulder_bags, tote_bags, travel_bags
  eyewear   sunglasses
  footwear  boots, flats, heels, sandals
  jewelry   bracelets, earrings, necklaces
filters: bag_closure, carry_method, category, ..., price, primary_color, sizes, ...
```

**Run** a search, the way the agent's search tool calls it:

In [ ]:
found = post(f"{CATALOG}/query/text", {
    "text": ["black dress for a wedding"],
    "categories": [],
    "filters": {"subcategory": ["dresses"], "price": {"max": 150}},
    "k": 3,
})
for p in found["products"]:
    print(f"{p['attributes']['similarity']:.3f}  {p['display_name']}  ${p['price']['amount']}")
d = found["diagnostics"]
print(f"\nmatched {d['source_result_count']} -> {d['after_filter_count']} after filters -> {d['returned_count']} returned")
print("filtered in Milvus:", d["filters_pushed_down"], "| in Python:", d["filters_decided_in_python"])

**You should see** ranked products, then how the filters narrowed the set:

```
0.727  Black Satin Lace-Up Dress  $69.99
0.721  Vivacious Velvet Dress  $129.99
0.710  Belle Noir Satin Gown  $129.99

matched 33 -> 23 after filters -> 3 returned
filtered in Milvus: ['subcategory'] | in Python: ['price']
```

**Try** an unknown value, such as `"subcategory": ["capes"]`. The catalog answers
HTTP 422 rather than quietly dropping the filter.

**Run** a product lookup by id:

In [ ]:
product_id = found["products"][0]["product_id"]
detail = get(f"{CATALOG}/products/{product_id}")
print(detail["display_name"], "-", detail["category"])
print({k: detail["attributes"][k] for k in list(detail["attributes"])[:6]})

**You should see** the product and its catalog attributes. These are the only
product facts the agent may state:

```
Black Satin Lace-Up Dress - dresses
{'category': 'apparel', 'closure': 'tie', 'composition': '100% satin', 'garment_length': 'full_length', ...}
```

### 6.3 memory-retriever: carts and conversations

Memory holds everything that outlives a turn. Because of that, any
chain-server replica can serve any turn. It listens on `127.0.0.1`, so this
machine can reach it and the browser cannot.

**Run:**

In [ ]:
cart = get(f"{MEMORY}/user/{NOTEBOOK_USER_ID}/cart")
for line in cart["cart"]:
    print(line["cart_line_id"], line["item"], "size", line.get("size"), "x", line["amount"], f"${line['price']}")

**You should see** the dress from step 5:

```
c2407c7360a2413fa5266c5ebbde5473 Black Satin Lace-Up Dress size 6 x 1 $69.99
```

This is the authoritative cart. The agent's reply is not. When the two
disagree, the cart is right, and it is what evaluation checks.

Other routes: `/user/{id}/cart/add|remove|clear`,
`PUT /user/{id}/cart/{line}/quantity`, and `/conversations/{id}/...` for turn
records and for resolving "the first one". See [docs/API.md](../docs/API.md).

### 6.4 Inside the agent: skills, tools, and the gate

Skills are Markdown files in `chain_server/skills/shopper/`. Each turn, the
agent activates the skills that fit. Each skill's header lists the tools it
grants, and a tool that no active skill grants is refused. This **skill gate**
is why a styling question cannot write to the cart.

**Run:**

In [ ]:
for skill in sorted((REPO / "chain_server/skills/shopper").glob("*/SKILL.md")):
    header = skill_header(skill)
    tools = [t.removesuffix("_tool") for t in header.get("tools_granted", [])]
    print(f"{header['name']:21} {', '.join(tools) or '(guidance only)'}")

**You should see** one line per skill:

```
budget-shopping       (guidance only)
cart-management       get_cart, add_cart_items, remove_cart_item, update_cart_items, view_cart_total, ...
catalog-questions     describe_catalog, search_catalog, get_product_details
destination-weather   get_weather_forecast
outfit-styling        search_catalog, get_product_details, check_product_availability, ...
product-discovery     describe_catalog, search_catalog, get_product_details, ...
store-policy-answers  get_store_policy
```

One more rule sits on top. A cart write needs a skill that grants it to be
chosen at the **start** of the turn. A turn that began as "show me dresses"
cannot end by adding something the shopper never asked for.

A test keeps an exact copy of what the model reads: its tools, their
descriptions, and the system prompt. That copy is the quickest way to see the
tools as the model sees them.

**Run:**

In [ ]:
snapshot = json.loads((REPO / "tests/unit/chain_server/model_facing_snapshot.json").read_text())
for tool in snapshot["tools"]:
    print(f"{tool['name']:36} {tool['description'].splitlines()[0][:64]}")
print(f"\nsystem prompt: {len(snapshot['system_prompt'])} characters")

**You should see** 14 tools with the first line of each description, then the
prompt size:

```
activate_shopper_skills_tool         Select and load shopper behavior skills for this turn. This is
search_catalog_tool                  Find products by description, advertised taxonomy, or constraints.
...
add_cart_items_tool                  Add products to the cart. Use ONLY on explicit shopper intent to
...
```

Changing this text changes which tools the agent calls. The test fails until
the copy is refreshed on purpose, and an evaluation replay confirms the change.

## 7. When something is off

**Do** these first, in a terminal:

```bash
docker compose ps                          # anything restarting or exited?
docker compose logs --tail 50 <service>    # e.g. chain-server, catalog-retriever
```

| Symptom | Check | Cause and fix |
|---|---|---|
| `wait_until_ready` stuck on the catalog | `curl localhost:8010/ready` returns 503 | index not built yet, or the indexer failed. Read `docker compose logs catalog-indexer` |
| Replies say the catalog is empty | the same `/ready` | the same cause. `/health` stays 200 without an index, so always check `/ready` |
| Turns fail with `429` | `docker compose logs chain-server` | the hosted endpoint's rate limit. Send one conversation at a time |
| A turn is slow or surprising | its trace | open Phoenix at [localhost:6006](http://localhost:6006). See notebook 2 |

Every turn also returns its own diagnostics. Read them first when the agent
does something unexpected.

**Run:**

In [ ]:
d = turn["diagnostics"]
for call in d.get("tool_calls", []):
    print(call["sequence"], call["tool_name"], call["status"], call.get("rejection_reason") or "")
print("ended:", d.get("final_termination_reason"))

**You should see** each tool call of the last turn, in order, with its status:

```
1 activate_shopper_skills_tool completed
2 resolve_conversation_products_tool completed
3 add_cart_items_tool completed
ended: completed
```

A refused call shows `rejected` and the reason, for example a cart write that
no skill chosen at the start of the turn allowed.

## 8. Clean up

**Run** the next cell to clear this notebook's shopper: its cart and its context.

In [ ]:
clear_shopper()
print("cart:", get(f"{MEMORY}/user/{NOTEBOOK_USER_ID}/cart")["cart"])

**You should see** `cart: []`.

**Do**, when you are finished with the stack, in a terminal at the repo root:

```bash
docker compose down        # stop; the index and carts are kept
docker compose down -v     # stop and delete them too
```

**Next: [2 · Observability](2_Observability.ipynb).** Every turn you ran here is
now a trace in Phoenix. The next notebook reads them.